# 15.13 如何公平比較 Dense／MoE？


如果MoE多存了四套FFN，卻拿它與只有一套、相同寬度的Dense比較，品質提升可能只是來自更多參數。反過來，若兩者存一樣多，MoE每個token又可能用較少計算。公平比較必須先回答：我們想固定儲存容量，還是固定每個token的大致工作量？按[總權重](https://birdhackor.github.io/tiny-perceptron-vlm/15.10.html)匹配，和按每次使用估計匹配，不會自然產生同一個Dense尺寸；[實測時間](https://birdhackor.github.io/tiny-perceptron-vlm/15.11.html)還要另外列出。先用四位小expert把兩種尺寸各算一次。

用不含bias的玩具尺寸推算：特徵寬度4，每位expert中間寬度2，單位有 `4×2+2×4=16` 個權重。四位共64，router另有16，總共80。若Dense也要80個權重，它的中間寬度應是10，因為 `4×10+10×4=80`。但MoE若top-1，每token使用估計只有 `16+16=32`；按這個估計配Dense，寬度4就足夠。兩種「匹配」得到10和4，回答的是不同問題。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
width, expert_hidden, experts = 4, 2, 4
per_expert = 2 * width * expert_hidden
router = width * experts
total = experts * per_expert + router
for k in (1, 2):
    active = k * per_expert + router
    same_parameters_hidden = total / (2 * width)
    same_active_estimate_hidden = active / (2 * width)
    print("k", k, "MoE總權重", total, "使用估計", active)
    print("同總參數Dense寬度", same_parameters_hidden, "同使用估計Dense寬度", same_active_estimate_hidden)

程式輸入都是架構尺寸，沒有訓練或計時。k=1時總權重80、使用估計32，兩個Dense寬度10與4；k=2時總權重仍80，使用估計48，寬度10與6。這裡結果剛好是整數，真實尺寸可能需要取整並報出剩餘差距。你應從輸出看見top-k影響工作量，不影響已存的expert套數。

按使用權重數匹配只是計算成本的初步代理，不能直接稱為精確FLOPs匹配。FLOPs是浮點運算次數，還需把router、啟動函數、合併，以及完整模型的attention等納入。即使運算次數接近，索引與搬移成本仍可能讓時間不同，所以應另列實測耗時、最高記憶體占用與[路由負載](https://birdhackor.github.io/tiny-perceptron-vlm/15.8.html)：各expert被選次數，以及次數占總分派筆數的比例。參數匹配則應數實際模組的全部參數，包含bias及共享部分，不能只沿用此簡化算式。

完成尺寸選擇後，兩個模型還應使用相同訓練資料、相同已讀token數、相同[驗證集](https://birdhackor.github.io/tiny-perceptron-vlm/5.10.html)，也就是留作比較方案、其誤差不直接用來更新權重的資料。再重複幾個隨機種子：種子指定亂數起點，讓初始化可重現；換幾個起點能檢查差異是否只是偶然。若只用不同資料訓練各一次，結果混入資料與偶然性，無法歸因架構。最後分別報「同總參數」和「同計算預算」的比較，讓讀者知道結論在什麼資源條件下成立。

一輪TinyStories小模型比較，得到下面三個結果。這裡的每token容量代理，把全部非expert參數，加上每層選中expert的參數；它連整張字表也列入，並不是每個token實際讀取的格數或精確FLOPs。

| 比較版本 | 總參數 | 每token容量代理 | 驗證代價 |
| --- | ---: | ---: | ---: |
| Dense寬80：近似匹配MoE容量代理 | 207,680 | 207,680 | 2.28200 |
| Dense寬104：近似匹配MoE總參數 | 329,888 | 329,888 | 2.22603 |
| MoE top-2，輔助係數0.01 | 340,608 | 208,256 | 2.27819 |

代價是下一byte／故事結束目標的平均負log機率，越低表示在這批留出原文上較容易預測。MoE對前一種基準略低，對後一種基準較高：固定哪種資源，會改變結論。原比較七組各更新180次、讀337,761個有效訓練目標，驗證39,256個目標；只有seed 42一輪，不能把微小差距當成穩定優勢。完整七組、最後檢查與近似匹配差距留在選讀。

練習只把experts由4改成8，保留原來的兩輪迴圈；這次先只比較印出k=1的那一輪。先手算總權重160、使用估計48，同總參數Dense寬度20、同使用估計寬度6，再執行核對。程式也會另外印出k=2的結果，請依前面的公式分清兩輪使用的是不同k。這也提醒router隨expert數變大，計算匹配時不能假設它始終免費。

<details>
<summary>選讀：來源、量測條件與原始實報</summary>

我們依[T.8](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.8)的完整TinyStories切分，實際做了兩種匹配。Dense寬度在16到160的四倍數中，選總參數最接近目標者：匹配top-1代理142,080得到寬度64、141,568個，少512；匹配top-2代理208,256得到寬度80、207,680個，少576；匹配MoE總數340,608則是寬度104、329,888個，少10,720（約3.15%）。這些是近似匹配；改完整模型寬度也改變attention與字表形狀，並非只調FFN，更不能共用全部初始化。

| 模型 | 總參數 | 每token容量代理 | 驗證代價 | 最後檢查代價 |
| --- | ---: | ---: | ---: | ---: |
| Dense，匹配top-1代理 | 141,568 | 141,568 | 2.29999 | 2.32093 |
| Dense，匹配top-2代理 | 207,680 | 207,680 | 2.28200 | 2.30906 |
| Dense，匹配MoE總數 | 329,888 | 329,888 | 2.22603 | 2.24943 |
| MoE top-1，係數0 | 340,608 | 142,080 | 2.30115 | 2.32369 |
| MoE top-1，係數0.01 | 340,608 | 142,080 | 2.29775 | 2.31963 |
| MoE top-2，係數0 | 340,608 | 208,256 | 2.28011 | 2.30320 |
| MoE top-2，係數0.01 | 340,608 | 208,256 | 2.27819 | 2.30170 |

七組各完成180次更新、337,761個有效訓練目標，驗證與最後檢查分別計39,256、41,914個目標，包含各篇結束。top-2、0.01比代理相近的Dense略低，比總容量相近的Dense卻高；固定哪種資源，會得到不同比較。這只是seed 42的一輪短訓，差異沒有多種子的不確定性估計，代理也不是相同FLOPs。速度與記憶體另見[15.11](https://birdhackor.github.io/tiny-perceptron-vlm/15.11.html)。

我們固定保存top-2、0.01為後續教學教師，沒有按驗證成績選最好的一列。它在第一筆最後檢查提示 `Once upon a time, there ` 上仍生成 `tthe the the the the the the the`；代價略低並不等於有可用故事能力。[實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/moe.json)保存完整留出代價、每側前八篇生成、梯度與每層負載，讀者可核對上述取捨。

</details>
